# Figure panels for hit model fits across datasets

## Imports

Calcium tracks data processing functions are in the `analysis/tracks.jl` module. Functions that make final, detailed plots are in `analysis/tracksPlots.jl`. Functions to fit the different hit models are in `analysis/modelsML.jl`. 

In [ ]:
using Plots: Plots as pl
using Printf: @sprintf
using Measures: mm



# Load the data processing module
module tracksAnalysis
    include("analysis/tracks.jl")
end
using .tracksAnalysis: prepalldsets, loadalldsets, threshalldsets, 
        prepare_all_cdfs, get_csv_file_names

module hitModels
    include("analysis/hitModelsML.jl")
end
using .hitModels: onehit_cdf, onehit_loglhood_total,
    twohit_cdf, twohit_loglhood_total,
    threehit_cdf, threehit_loglhood_total, fithitmodel
    

module tracksPlots
    include("analysis/tracksPlots.jl")
end
using .tracksPlots: plotrobustness, plot_ml_cdf_comparison, plot_single_dataset_cdf


In [ ]:
# Default styles
pl.default(
    labelfontsize = 14,
    tickfontsize = 12,
    framestyle = :axes,  # https://docs.juliaplots.org/stable/gallery/gr/generated/gr-ref034/
    yminorgrid = false,
    xminorgrid = false,
    xgrid = false,
    ygrid = false,
    size = (550, 450),
    linewidth = 2.5,
    legendfontsize=12, 
    fontfamily="Arial"
)

# Default analysis parameters
default_fluo_factor = 0.8
cdfthreshold = 0.75
# Filtering parameters: maximal start time, minimal traj length
minlength = 50
maxstarttime = 30.0
filter_params = [minlength, maxstarttime]

# Fit each model across datasets

In [ ]:
### MAIN ANALYSIS FUNCTION ###

function fit_ml_all_datasets(; 
    datadir="data/JR064_tracks",
    plotdir="figures/final_tracks_panels/",
    fracabove=default_fluo_factor,
    cdfthresh=cdfthreshold)
    
    # Create output directory if it doesn't exist
    if !isdir(plotdir)
        mkpath(plotdir)
    end
    
    # Get a cleaned up list of data file names and paths
    fnames, fpaths = get_csv_file_names(datadir)
    
    println("MAXIMUM LIKELIHOOD FITTING - JR064 DATASETS")
    println("\nLoading and preprocessing datasets...")
    
    # Step 1: Preprocess all datasets
    tactiv_alldsets, durations, nicenames = prepalldsets(fpaths; 
        fracabove=fracabove, disp=false, filterpms=filter_params)
    
    n_dsets = length(tactiv_alldsets)
    println("\nLoaded $n_dsets datasets:")
    for (i, name) in enumerate(nicenames)
        println("  [$i] $name")
    end
    
    # Step 2: Fit the one-hit model
    println("Fitting ONE-HIT model (overall k2, one kc and one t0 per dataset)...")
    onehit_result, onehit_all_rms = fithitmodel(tactiv_alldsets, 
                    nicenames, durations, cdfthresh, 
                    onehit_loglhood_total, onehit_cdf)
    onehit_params = onehit_result[1:end-1]
    onehit_k2 = onehit_params[1]
    onehit_kcs = onehit_result[2:1+n_dsets]
    onehit_t0s = onehit_result[2+n_dsets:end-1]
    onehit_rms = onehit_result[end]
    
    println("\nOne-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" onehit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(onehit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] onehit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end
    
    # Step 3: Fit the two-hit model
    println("Fitting TWO-HIT model (overall k2, one kc and one t0 per dataset)...")
    twohit_result, twohit_all_rms = fithitmodel(tactiv_alldsets, 
                        nicenames, durations, cdfthresh,
                        twohit_loglhood_total, twohit_cdf)
    twohit_params = twohit_result[1:end-1]
    twohit_k2 = twohit_params[1]
    twohit_kcs = twohit_result[2:1+n_dsets]
    twohit_t0s = twohit_result[2+n_dsets:end-1]
    twohit_rms = twohit_result[end]
    
    println("\nTwo-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" twohit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(twohit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] twohit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end
    
    # Step 4: Fit the three-hit model
    println("Fitting THREE-HIT model (overall k2, one kc and one t0 per dataset)...")
    threehit_result, threehit_all_rms = fithitmodel(tactiv_alldsets, 
                            nicenames, durations, cdfthresh,
                            threehit_loglhood_total, threehit_cdf)
    threehit_params = threehit_result[1:end-1]
    threehit_k2 = threehit_params[1]
    threehit_kcs = threehit_result[2:1+n_dsets]
    threehit_t0s = threehit_result[2+n_dsets:end-1]
    threehit_rms = threehit_result[end]
    
    println("\nThree-hit model results:")
    println(@sprintf "  k2        = %.4f s^-1" threehit_k2)
    println("  t0 per dataset:")
    for (i, t0) in enumerate(threehit_t0s)
        println(@sprintf "    Dataset %d (%s):  kc = %.4f s^-1" i nicenames[i] threehit_kcs[i])
        println(@sprintf "    Dataset %d (%s): t0 = %.4f s" i nicenames[i] t0)
    end

    # Step 5: Compute and compare RMS errors
    println("RMS ERROR COMPARISON")

    println("RMS per dataset:")
    for (i, t0) in enumerate(threehit_t0s)
        println(@sprintf "\n    Dataset %d (%s), one-hit = %.6e" i nicenames[i] onehit_all_rms[i])
        println(@sprintf "    Dataset %d (%s), two-hit = %.6e" i nicenames[i] twohit_all_rms[i])
        println(@sprintf "    Dataset %d (%s), three-hit = %.6e" i nicenames[i] threehit_all_rms[i])
    end
    
    # Step 6: Create comparison plots
    println("Generating plots...")

    # Prepare all empirical CDFs
    prep_cdf_res = prepare_all_cdfs(tactiv_alldsets, nicenames, durations, cdfthresh)
    
    # Plot: All datasets, all models
    # For information, not for the final figure. 
    p_all = plot_ml_cdf_comparison(prep_cdf_res, nicenames, 
                                    onehit_params, twohit_params, threehit_params,
                                    onehit_cdf, twohit_cdf, threehit_cdf)
    plot_file = joinpath(plotdir, "ml_cdf_all_datasets_comparison.pdf")
    pl.savefig(p_all, plot_file)
    println("  Saved: $plot_file")
    display(p_all)

    # Plot: one dataset, for illustration of the fit procedure in the figure
    # At high concentrations, k2 determines the fit, and it's not perfect
    # because it is fitted across replicates which are not exactly identical
    # Focus on lower concentrations where we are comparing the fit quality. 
    i_show = 7  # Choice of dataset to show
    prep_cdf_i = [prep_cdf_res[j][i_show] for j in eachindex(prep_cdf_res)]
    k2_1, kc_1, t0_1 = onehit_params[1], onehit_params[1+i_show], onehit_params[1+n_dsets+i_show]
    k2_2, kc_2, t0_2 = twohit_params[1], twohit_params[1+i_show], twohit_params[1+n_dsets+i_show]
    k2_3, kc_3, t0_3 = threehit_params[1], threehit_params[1+i_show], threehit_params[1+n_dsets+i_show]
    
    p_single = plot_single_dataset_cdf(
        prep_cdf_i, nicenames[i_show], durations[i_show], cdfthresh,
        kc_1, k2_1, t0_1, kc_2, k2_2, t0_2, kc_3, k2_3, t0_3,
        onehit_all_rms[i_show], twohit_all_rms[i_show], threehit_all_rms[i_show],
        onehit_cdf, twohit_cdf, threehit_cdf
    )
    dsetname = replace(nicenames[i_show], "." => "-")
    plot_file = joinpath(plotdir, "ml_cdf_fits_dataset_$(dsetname).pdf")
    pl.savefig(p_single, plot_file)
    display(p_single)
    println("  Saved: $plot_file")

    println("Analysis complete!")
    
    return (onehit_params, twohit_params, threehit_params, 
            onehit_rms, twohit_rms, threehit_rms,
            nicenames, tactiv_alldsets)
end

In [ ]:
fit_ml_all_datasets();

# Robustness analysis
Wrap in a main function, define plotting functions above (improved versions of the PLots-based ones in the scripts). 

In [ ]:

# Robustness analysis
# as we vary the fluorescence threshold
# Main analysis + plotting function for varying the fluorescence threshold
function robustness_vs_thresh_main(; 
    datadir="data/JR064_tracks",
    plotdir="figures/final_tracks_panels/",
    fracabove_default=default_fluo_factor,
    cdfthresh=cdfthreshold, 
    dosave=true)
    
    # Get a cleaned up list of data file names and paths
    fnames, fpaths = get_csv_file_names(datadir)

    nfluopts = 50
    
    # Save CDF fits for each file and each threshold fraction tried
    # number of parameters per dataset: one k2 plus (kc, t0) for each dset
    n_dsets = length(fnames)
    all_p1 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms1 = zeros(Float64, nfluopts, n_dsets)
    all_p2 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms2 = zeros(Float64, nfluopts, n_dsets)
    all_p3 = zeros(Float64, nfluopts, 1 + 2*n_dsets)
    all_rms3 = zeros(Float64, nfluopts, n_dsets)
    
    fluothreshrange = collect(LinRange(0.1, 1.5, nfluopts))

    # Load data once
    tracks_alldsets, nicenames = loadalldsets(fpaths; disp=false)

    # Repeat analysis for a range of fluorescence thresholds
    for j in 1:1:nfluopts
        tactiv_alldsets, durations = threshalldsets(
            tracks_alldsets, nicenames, fluothreshrange[j]; disp=false)
    
        # Fit the one-hit, two-hit, and three-hit models
        p1, rms1 = fithitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    onehit_loglhood_total, onehit_cdf)
        all_p1[j, :] = p1[1:end-1]  # last entry is RMS
        all_rms1[j, :] = rms1

        p2, rms2 = fithitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    twohit_loglhood_total, twohit_cdf)
        all_p2[j, :] = p2[1:end-1]
        all_rms2[j, :] = rms2

        p3, rms3 = fithitmodel(tactiv_alldsets, nicenames, durations, cdfthresh, 
                    threehit_loglhood_total, threehit_cdf)
        all_p3[j, :] = p3[1:end-1]
        all_rms3[j, :] = rms3
        println(@sprintf "Finished fits for fluorescence threshold factor = %.2f" fluothreshrange[j]+1.0)
    end

    # Plot slope and goodness of linear vs quadratic fits
    # as a function of the threshold fraction
    # linfits, quadfits, allnames, fracrange, fname
    all_params = (all_p1, all_p2, all_p3)
    all_rms = (all_rms1, all_rms2, all_rms3)

    allplots = plotrobustness((all_p1, all_p2, all_p3), 
                    (all_rms1, all_rms2, all_rms3), fnames, fluothreshrange; 
                    fluothresh_default=fracabove_default)
    
    # Display and save the resulting plots
    pp1, pp2, pp3 = allplots

    fname = joinpath(plotdir, "paramfits_kc_vs_threshold.pdf")
    display(pp1)
    if dosave
        pl.savefig(pp1, fname)
    end

    fname = joinpath(plotdir, "onehit_vs_twohit_fitRMS.pdf")
    display(pp2)
    if dosave
        pl.savefig(pp2, fname)
    end

    fname = joinpath(plotdir, "best12_vs_3-hit_fitRMS.pdf")
    display(pp3)
    if dosave
        pl.savefig(pp3, fname)
    end
    return 0
end

In [ ]:
robustness_vs_thresh_main();